In [1]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# 第一步：从SQL备份文件中提取logs数据
# ============================================================================

def parse_sql_insert_statements(sql_file_path, table_name='logs'):
    """
    解析SQL文件中的INSERT语句，提取数据
    """
    print(f"正在解析SQL文件: {sql_file_path}")
    print("这可能需要几分钟时间，请耐心等待...")
    
    data = []
    
    with open(sql_file_path, 'r', encoding='utf-8') as f:
        buffer = ""
        insert_pattern = f"INSERT INTO `{table_name}` VALUES "
        
        for line_num, line in enumerate(f, 1):
            if line_num % 100000 == 0:
                print(f"  已处理 {line_num} 行...")
            
            # 跳过注释和空行
            if line.startswith('--') or line.startswith('/*') or line.startswith('*/') or not line.strip():
                continue
            
            buffer += line
            
            # 查找INSERT语句
            if insert_pattern in buffer:
                # 提取VALUES后面的内容
                try:
                    values_start = buffer.find(insert_pattern) + len(insert_pattern)
                    values_section = buffer[values_start:]
                    
                    # 使用正则表达式提取所有的值元组
                    # 匹配形式：(value1,value2,...,value10)
                    pattern = r'\(([^)]+)\)'
                    matches = re.findall(pattern, values_section)
                    
                    for match in matches:
                        # 分割每个值
                        values = []
                        current = ""
                        in_quotes = False
                        quote_char = None
                        
                        for char in match:
                            if char in ('"', "'") and (not in_quotes or char == quote_char):
                                if in_quotes and char == quote_char:
                                    in_quotes = False
                                    quote_char = None
                                elif not in_quotes:
                                    in_quotes = True
                                    quote_char = char
                                current += char
                            elif char == ',' and not in_quotes:
                                values.append(current.strip().strip('"').strip("'"))
                                current = ""
                            else:
                                current += char
                        
                        if current:
                            values.append(current.strip().strip('"').strip("'"))
                        
                        if len(values) == 10:  # logs表有10列
                            data.append(values)
                    
                    buffer = ""
                    
                except Exception as e:
                    print(f"警告: 第{line_num}行解析出错: {e}")
                    buffer = ""
                    continue
    
    print(f"✓ SQL文件解析完成，共提取 {len(data)} 条记录")
    
    # 创建DataFrame
    columns = ['id', 'user_id', 'qid', 'docno', 'event_type', 
               'start_idx', 'end_idx', 'duration', 'pass_flag', 'timestamp']
    df = pd.DataFrame(data, columns=columns)
    
    # 转换数据类型
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    df['qid'] = pd.to_numeric(df['qid'], errors='coerce')
    df['start_idx'] = pd.to_numeric(df['start_idx'], errors='coerce')
    df['end_idx'] = pd.to_numeric(df['end_idx'], errors='coerce')
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
    df['pass_flag'] = pd.to_numeric(df['pass_flag'], errors='coerce')
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    return df

In [5]:
# ============================================================================
# 第二步：主程序
# ============================================================================

print("=" * 80)
print("用户行为质量分析系统 - SQL直接解析版")
print("=" * 80)
print()

# 从SQL文件中提取logs数据
sql_file = 'backup_qe_v2_full.sql'
logs_df = parse_sql_insert_statements(sql_file, table_name='logs')

print()
print(f"数据加载完成：{len(logs_df)} 条日志记录")
print(f"包含用户数量：{logs_df['user_id'].nunique()} 个")
print()

# 显示数据概览
print("数据概览（前5行）：")
print(logs_df.head())
print()

print("事件类型分布：")
print(logs_df['event_type'].value_counts())
print()

# 读取interleaving结果数据，筛选Easy Negative文档
print("正在加载Easy Negative文档数据...")
try:
    # 读取interleaving_results.csv
    interleaving_df = pd.read_csv('result_preference_based_with_text_gold.csv')
    print(f"✓ Interleaving结果数据加载完成：{len(interleaving_df)} 条记录")
    
    # 只保留origin_label为'Easy-Negative'的文档
    easy_neg_df = interleaving_df[interleaving_df['origin_label'] == 'Easy-Negative'].copy()
    print(f"✓ 筛选出Easy Negative文档：{len(easy_neg_df)} 条记录")
    
    # 显示origin_label的分布
    print("\norigin_label分布：")
    print(interleaving_df['origin_label'].value_counts())
    
except FileNotFoundError:
    print("警告：未找到 interleaving_results.csv 文件")
    print("将创建空的DataFrame，Easy Negative分析将跳过")
    easy_neg_df = pd.DataFrame(columns=['qid', 'docno', 'origin_label'])

print()

用户行为质量分析系统 - SQL直接解析版

正在解析SQL文件: backup_qe_v2_full.sql
这可能需要几分钟时间，请耐心等待...
✓ SQL文件解析完成，共提取 48881 条记录

数据加载完成：48881 条日志记录
包含用户数量：215 个

数据概览（前5行）：
   id                   user_id   qid                         docno  \
0  74  6357a92c14a05d21ebdd897f  2082                             0   
1  75  627286792f25495f123cc27b  2082                             0   
2  76  5c9d88bb2b3c77001744ec1c  2082                             0   
3  77  6357a92c14a05d21ebdd897f  2082  msmarco_passage_39_461879190   
4  78  666a1e606b80f9865c1f3d5f  2082                             0   

      event_type  start_idx  end_idx  duration  pass_flag           timestamp  
0  BOT_DETECTION         -1       -1      7580          1 2025-04-21 12:26:53  
1  BOT_DETECTION         -1       -1     12339          1 2025-04-21 12:27:41  
2  BOT_DETECTION         -1       -1      9491          1 2025-04-21 12:27:45  
3       OPEN_DOC         -1       -1     55110          0 2025-04-21 12:27:48  
4  BOT_DETECTION         -

In [6]:
# ============================================================================
# 第三步：识别最近的27个活跃用户（或指定用户列表）
# ============================================================================

print("=" * 80)
print("识别目标用户")
print("=" * 80)
print()

# 方法1：手动指定27个用户ID（如果你已经知道）v1上半
'''MANUAL_USERS = ['5b68c9eb87af310001584803',
                '55d51a6b8ce09000127d4821',
                '67181a55422d48f5727f08ec',
                '595cd88562431800019d691a',
                '610c66163e803a3564dfaf9c',
                '66435e2e5c5b5aceb2195908',
                '628f572a2d1304a34b9e1a37',
                '65de30b34b701c678a13c75b',
                '5f540fc665e3b6148c04f10e',
                '5f1088e27a43fe0c64ca5814',
                '66cc6eb364154a51d4d44eb1',
                '66cec5a3fdf1fe2c010e9971',
                '5c6e60b04e08ad00018cc995',
                '673e222bdb1a35935b3075f1',
                '659c067de92634488b349626',
                '5d68c8aa40524c00189e8ac2',
                '66fa1820329fa9bb88f2e3c6',
                '6012e262527c380ccb2a6c74',
                '668a9e53e7690b3b64d48e92',
                '5ca102f57114700016da340d',
                '5c6709ab926a5b0001eb29c1',
                '66d9cd06e207a93377f7820e',
                '6151e479b4ee16668948c7f2',
                '5d542c3a5af5900019bc80c4',
                '5f565a2b33c152071057bb04',
                '62d6d77df59450e545fb22f6',
                '63cedb764240865b7177c745']  # 在这里填入用户ID列表'''
#v1下半的用户id
'''MANUAL_USERS = ['66acbaf5c803ca1a1ef3f056',
                '669053bbebafdd430e770142',
                '660b2346862ff2a3da9cfe3e',
                '6413a4a1a6e064ffb173f1ce',
                '67ddd4f85ab870ffd90cc888',
                '5bbb5e3f118c0600013c3202',
                '5d8b66f5d189bd001a378273',
                '5a143884bd04860001b01123',
                '607864ca34af93e712224f01',
                '66b73f9ba035651170b94c76',
                '5c81582c4c1a61000155584f',
                '5c4f5967aac8be0001716a65',
                '5c0676a61a20110001ea1a24',
                '615eda024a5f84f408625b4f',
                '682c3f62f2680e9a67130379',
                '6756d530acef7d25ac1164c8',
                '67a95d468d14bd6324b4bbba',
                '5ca5189dce3f920016239854',
                '5d8e154af2858200171fdb95',
                '5bdcf08f05ed110001ab2bbf',
                '66e4c4358de330a06b4d33f1',
                '668bf688d58206e61f399a88',
                '67cf0c70aa43b13b6168ecbf',
                '5f392fae9168cc472f0468f6',
                '6658c56f3d9230d99c3e6752',
                '63fbf0e3b18cc14adc0dbfb6',
                '647f87dfe5161fce9adc1b97',
                '5bb7abf3c190e20001e5c2c8']'''
#v2上半的用户id
'''MANUAL_USERS = ['67842c60c9cba1e154a69bbd',
                '63f0ac76a8c1914f6f163539',
                '66f5617d6ea358aa9f275019',
                '5c7ceda01d2afc0001f4ad1d',
                '62fb8e94ce94179d6a4e8277',
                '6641059cdec71f7b7d5a5e2b',
                '665a222ec21c09f2b8731fde',
                '63ea4d64095b3286433ed52e',
                '5b490405bc06c90001f8d6a5',
                '671d39090c9e2ad9ec75a03c',
                '66db9db4324609f1a7231f49',
                '5ebaa6dd28b52b000c94b78c',
                '5f88877885e51d02acd71b12',
                '5ef89ca507b0e3040299f1a3',
                '66aa95c94ab0bbc1be429c3e',
                '64480b662313e1b5ec1c7d57',
                '5d37c0d61b03b5000151950b',
                '6100372e3a116cb5cb36ebea',
                '60ac66a8e1bf5a1c51fa864c',
                '60719f4b18e2b30c9e5a562d',
                '5fc98fe1b105424d4b03381b',
                '67ef32ae3b9945398e55b406',
                '5a9e9fc46219a30001f54994',
                '5d0dca2cecd8f60017eaddb3',
                '66019f5e10adbe524f2a844a',
                '5df7b6b6ef7f5e55506510c7',
                '5f53b958c8cfea6e2104c5b6']'''

MANUAL_USERS = ['5a9fa90f873cda00010d29d8',
                '67e85a8becc5f7048616c4be',
                '679153802a458f825959cf4a',
                '5a0c4184fe645f0001e9f5dd',
                '5d31dc6e42678e001a0bdedd',
                '5589b65afdf99b118adfa595',
                '664dad1672f2f70e4aa2a012',
                '6601a50336e8fbd902820049',
                '5c4d8d706b23ab0001a34af2',
                '5e3a9e0dfbce8a296258c297',
                '581ccd016c73180001fa5b2a',
                '677bccba65784d9a5a876468',
                '662e3e72d494903379b6956a',
                '6637b21240220c2517fffa1a',
                '6626aca000aab07fdd4147c0',
                '65f860a1ab03181926bce465',
                '660453c9877e571d12b555f7',
                '5d31dc6e42678e001a0bdedd',
                '60b542b8fbb1a6a2678f5054',
                '5ce42b3454e33d001918923a',
                '63106ea4e8fe9c6cd32bf358',
                '5a70b6b882968f0001a6b12f',
                '67f2fbc77b115d1edb94ef11',
                '66afcaa60f7d8f58dc21db8e',
                '667b24c6c25f7a12d08d9ab6',
                '66ac261efa487682e38c3024',
                '65e4a6bdde84e615dcac2651']

# 方法2：自动选择最活跃的27个用户
if len(MANUAL_USERS) == 0:
    print("未指定用户列表，将自动选择最活跃的27个用户")
    
    # 统计每个用户的活动情况
    user_stats = logs_df.groupby('user_id').agg({
        'id': 'count',
        'timestamp': ['min', 'max']
    }).reset_index()
    user_stats.columns = ['user_id', 'log_count', 'first_activity', 'last_activity']
    
    # 按最后活动时间排序，选择最近的27个用户
    user_stats = user_stats.sort_values('last_activity', ascending=False)
    target_users = user_stats.head(27)['user_id'].tolist()
    
    print("最活跃的27个用户：")
    print(user_stats.head(27)[['user_id', 'log_count', 'last_activity']])
else:
    target_users = MANUAL_USERS
    print(f"使用手动指定的 {len(target_users)} 个用户")

print()
print(f"目标用户数量：{len(target_users)}")
print()

# 筛选目标用户的日志
filtered_logs = logs_df[logs_df['user_id'].isin(target_users)].copy()
print(f"筛选后的日志数据：{len(filtered_logs)} 条记录")
print()

识别目标用户

使用手动指定的 27 个用户

目标用户数量：27

筛选后的日志数据：7723 条记录



In [7]:
# ============================================================================
# 第三步之二：时间筛选（过滤旧实验数据）⭐ 重要
# ============================================================================

print("时间筛选（过滤旧实验数据）...")

# 方法1：指定当前实验的开始日期
EXPERIMENT_START_DATE = '2025-11-01'  # 修改为你的实验开始日期

# 方法2：或者直接设置为 None 跳过时间筛选
# EXPERIMENT_START_DATE = None

if EXPERIMENT_START_DATE is not None:
    # 转换为datetime
    filtered_logs['timestamp'] = pd.to_datetime(filtered_logs['timestamp'], errors='coerce')
    experiment_start = pd.to_datetime(EXPERIMENT_START_DATE)
    
    before_time_filter = len(filtered_logs)
    filtered_logs = filtered_logs[filtered_logs['timestamp'] >= experiment_start].copy()
    after_time_filter = len(filtered_logs)
    
    removed_count = before_time_filter - after_time_filter
    print(f"✓ 移除了 {removed_count} 条 {EXPERIMENT_START_DATE} 之前的旧记录")
    print(f"✓ 剩余记录：{after_time_filter}")
    
    # 显示时间范围
    if len(filtered_logs) > 0:
        min_time = filtered_logs['timestamp'].min()
        max_time = filtered_logs['timestamp'].max()
        print(f"✓ 数据时间范围：{min_time} 至 {max_time}")
else:
    print("⚠️  未启用时间筛选，使用所有历史数据")
    print("   这可能包含之前实验的数据！")

print()

时间筛选（过滤旧实验数据）...
✓ 移除了 0 条 2025-11-01 之前的旧记录
✓ 剩余记录：7723
✓ 数据时间范围：2025-12-01 13:45:53 至 2025-12-01 20:34:08



In [8]:
# ============================================================================
# 第四步：创建Easy Negative文档集合
# ============================================================================

easy_negative_set = set(
    zip(easy_neg_df['qid'].astype(str), easy_neg_df['docno'].astype(str))
)
print(f"Easy Negative文档数量：{len(easy_negative_set)} 个")
print()

Easy Negative文档数量：11 个



In [9]:
# ============================================================================
# 第五步：分析每个用户的行为
# ============================================================================

print("=" * 80)
print("用户行为分析")
print("=" * 80)
print()

results = []

for idx, user_id in enumerate(target_users, 1):
    print(f"分析用户 {idx}/{len(target_users)}: {user_id}", end='\r')
    
    user_logs = filtered_logs[filtered_logs['user_id'] == user_id]
    
    if len(user_logs) == 0:
        continue
    
    user_result = {'user_id': user_id}
    
    # 1. Bot检测通过率
    bot_detection_logs = user_logs[user_logs['event_type'] == 'BOT_DETECTION']
    total_bot_checks = len(bot_detection_logs)
    passed_bot_checks = len(bot_detection_logs[bot_detection_logs['pass_flag'] == 1])
    
    if total_bot_checks > 0:
        bot_pass_rate = (passed_bot_checks / total_bot_checks) * 100
    else:
        bot_pass_rate = 0
    
    user_result['bot_checks_total'] = total_bot_checks
    user_result['bot_checks_passed'] = passed_bot_checks
    user_result['bot_pass_rate'] = bot_pass_rate
    
    # 2. 打开文档数量
    open_doc_logs = user_logs[user_logs['event_type'] == 'OPEN_DOC']
    open_doc_count = len(open_doc_logs)
    user_result['open_doc_count'] = open_doc_count
    
    # 3. 选择passage数量（原始）
    passage_selection_logs = user_logs[user_logs['event_type'] == 'PASSAGE_SELECTION']
    raw_passage_selection_count = len(passage_selection_logs)
    
    # 4. 取消行为数量（详细分类）
    cancel_doc_logs = user_logs[user_logs['event_type'] == 'CANCEL_DOC']
    total_cancel_count = len(cancel_doc_logs)
    
    # 4a. Cancel without passage selection (start_idx=-1 and end_idx=-1)
    # 这是打开了文档但没有选择passage就取消了
    cancel_without_selection = cancel_doc_logs[
        (cancel_doc_logs['start_idx'] == -1) & (cancel_doc_logs['end_idx'] == -1)
    ]
    cancel_without_selection_count = len(cancel_without_selection)
    
    # 4b. Cancel after passage selection (start_idx!=-1 or end_idx!=-1)
    # 这是选择了passage但又取消了
    cancel_after_selection = cancel_doc_logs[
        (cancel_doc_logs['start_idx'] != -1) | (cancel_doc_logs['end_idx'] != -1)
    ]
    cancel_after_selection_count = len(cancel_after_selection)
    
    # 最终的passage选择数量 = 原始选择数 - 取消的选择数
    final_passage_selection_count = raw_passage_selection_count - cancel_after_selection_count
    
    user_result['raw_passage_count'] = raw_passage_selection_count
    user_result['cancel_after_selection'] = cancel_after_selection_count
    user_result['final_passage_count'] = final_passage_selection_count
    user_result['cancel_without_selection'] = cancel_without_selection_count
    user_result['total_cancel_count'] = total_cancel_count
    
    # 5. 点击Easy Negative文档的次数
    easy_neg_clicks = 0
    for _, log in open_doc_logs.iterrows():
        qid_str = str(int(log['qid'])) if pd.notna(log['qid']) else '0'
        docno_str = str(log['docno'])
        if (qid_str, docno_str) in easy_negative_set:
            easy_neg_clicks += 1
    
    user_result['easy_negative_clicks'] = easy_neg_clicks
    
    # 6. 打开文档的平均duration
    if open_doc_count > 0:
        avg_open_duration = open_doc_logs['duration'].mean()
    else:
        avg_open_duration = 0
    user_result['avg_open_duration_ms'] = avg_open_duration
    user_result['avg_open_duration_sec'] = avg_open_duration / 1000 if avg_open_duration > 0 else 0
    
    # 7. 选择passage的平均duration
    if raw_passage_selection_count > 0:
        avg_passage_duration = passage_selection_logs['duration'].mean()
    else:
        avg_passage_duration = 0
    user_result['avg_passage_duration_ms'] = avg_passage_duration
    user_result['avg_passage_duration_sec'] = avg_passage_duration / 1000 if avg_passage_duration > 0 else 0
    
    # 8. 取消passage的平均duration（如果有的话）
    if cancel_after_selection_count > 0:
        avg_cancel_passage_duration = cancel_after_selection['duration'].mean()
    else:
        avg_cancel_passage_duration = 0
    user_result['avg_cancel_passage_duration_ms'] = avg_cancel_passage_duration
    user_result['avg_cancel_passage_duration_sec'] = avg_cancel_passage_duration / 1000 if avg_cancel_passage_duration > 0 else 0
    
    # 额外指标
    if open_doc_count > 0:
        total_cancel_rate = (total_cancel_count / open_doc_count) * 100
        cancel_without_selection_rate = (cancel_without_selection_count / open_doc_count) * 100
        cancel_after_selection_rate = (cancel_after_selection_count / open_doc_count) * 100
        easy_neg_rate = (easy_neg_clicks / open_doc_count) * 100
    else:
        total_cancel_rate = 0
        cancel_without_selection_rate = 0
        cancel_after_selection_rate = 0
        easy_neg_rate = 0
    
    user_result['total_cancel_rate'] = total_cancel_rate
    user_result['cancel_without_selection_rate'] = cancel_without_selection_rate
    user_result['cancel_after_selection_rate'] = cancel_after_selection_rate
    user_result['easy_neg_rate'] = easy_neg_rate
    
    # Passage完成率 = 最终passage数 / 原始passage数
    if raw_passage_selection_count > 0:
        passage_completion_rate = (final_passage_selection_count / raw_passage_selection_count) * 100
    else:
        passage_completion_rate = 0
    user_result['passage_completion_rate'] = passage_completion_rate
    
    results.append(user_result)

print()
print(f"✓ 完成 {len(results)} 个用户的行为分析")
print()

# 创建结果DataFrame
results_df = pd.DataFrame(results)

用户行为分析

分析用户 27/27: 65e4a6bdde84e615dcac2651
✓ 完成 27 个用户的行为分析



In [10]:
#============================================================================
# 第六步：计算汇总统计
# ============================================================================

print("=" * 80)
print("汇总统计")
print("=" * 80)
print()

summary = {
    '指标': [
        'Bot检测次数',
        'Bot检测通过次数',
        'Bot检测通过率 (%)',
        '打开文档总数',
        '原始Passage选择数',
        '取消的Passage选择数',
        '最终Passage选择数',
        '未选择就取消数',
        '总取消行为数',
        'Easy Negative点击总数',
        '平均打开文档时长 (秒)',
        '平均Passage选择时长 (秒)',
        '平均取消Passage时长 (秒)',
        '总取消率 (%)',
        '未选择取消率 (%)',
        '选择后取消率 (%)',
        'Easy Negative点击率 (%)',
        'Passage完成率 (%)'
    ],
    '总计': [
        results_df['bot_checks_total'].sum(),
        results_df['bot_checks_passed'].sum(),
        (results_df['bot_checks_passed'].sum() / results_df['bot_checks_total'].sum() * 100) 
            if results_df['bot_checks_total'].sum() > 0 else 0,
        results_df['open_doc_count'].sum(),
        results_df['raw_passage_count'].sum(),
        results_df['cancel_after_selection'].sum(),
        results_df['final_passage_count'].sum(),
        results_df['cancel_without_selection'].sum(),
        results_df['total_cancel_count'].sum(),
        results_df['easy_negative_clicks'].sum(),
        results_df['avg_open_duration_sec'].mean(),
        results_df['avg_passage_duration_sec'].mean(),
        results_df['avg_cancel_passage_duration_sec'].mean(),
        results_df['total_cancel_rate'].mean(),
        results_df['cancel_without_selection_rate'].mean(),
        results_df['cancel_after_selection_rate'].mean(),
        results_df['easy_neg_rate'].mean(),
        results_df['passage_completion_rate'].mean()
    ],
    '人均': [
        results_df['bot_checks_total'].mean(),
        results_df['bot_checks_passed'].mean(),
        results_df['bot_pass_rate'].mean(),
        results_df['open_doc_count'].mean(),
        results_df['raw_passage_count'].mean(),
        results_df['cancel_after_selection'].mean(),
        results_df['final_passage_count'].mean(),
        results_df['cancel_without_selection'].mean(),
        results_df['total_cancel_count'].mean(),
        results_df['easy_negative_clicks'].mean(),
        results_df['avg_open_duration_sec'].mean(),
        results_df['avg_passage_duration_sec'].mean(),
        results_df['avg_cancel_passage_duration_sec'].mean(),
        results_df['total_cancel_rate'].mean(),
        results_df['cancel_without_selection_rate'].mean(),
        results_df['cancel_after_selection_rate'].mean(),
        results_df['easy_neg_rate'].mean(),
        results_df['passage_completion_rate'].mean()
    ]
}

summary_df = pd.DataFrame(summary)
pd.options.display.float_format = '{:.2f}'.format
print(summary_df.to_string(index=False))
print()

汇总统计

                  指标      总计     人均
             Bot检测次数   27.00   1.00
           Bot检测通过次数   27.00   1.00
        Bot检测通过率 (%)  100.00 100.00
              打开文档总数 3583.00 132.70
        原始Passage选择数 3329.00 123.30
       取消的Passage选择数    0.00   0.00
        最终Passage选择数 3329.00 123.30
             未选择就取消数  272.00  10.07
              总取消行为数  272.00  10.07
   Easy Negative点击总数   54.00   2.00
        平均打开文档时长 (秒)   14.36  14.36
   平均Passage选择时长 (秒)    9.44   9.44
   平均取消Passage时长 (秒)    0.00   0.00
            总取消率 (%)    7.71   7.71
          未选择取消率 (%)    7.71   7.71
          选择后取消率 (%)    0.00   0.00
Easy Negative点击率 (%)    1.15   1.15
      Passage完成率 (%)  100.00 100.00



In [11]:
# ============================================================================
# 第七步：识别低质量工人
# ============================================================================

print("=" * 80)
print("低质量工人识别")
print("=" * 80)
print()

LOW_QUALITY_CRITERIA = {
    'bot_pass_rate': 90,
    'easy_neg_rate': 1,
    'total_cancel_rate': 50,
    'cancel_after_selection_rate': 1,  # 新增：选择后又取消的比例
    'avg_open_duration_sec': 5,
    'final_passage_count': 30,  # 改为最终passage数量
    'passage_completion_rate': 70  # 新增：passage完成率低于70%
}

results_df['is_low_quality'] = False
results_df['quality_issues'] = ''

for idx, row in results_df.iterrows():
    issues = []
    
    if row['bot_pass_rate'] < LOW_QUALITY_CRITERIA['bot_pass_rate']:
        issues.append(f"Low bot pass rate ({row['bot_pass_rate']:.1f}%)")
        results_df.at[idx, 'is_low_quality'] = True
    
    if row['easy_neg_rate'] > LOW_QUALITY_CRITERIA['easy_neg_rate']:
        issues.append(f"High easy-neg click rate ({row['easy_neg_rate']:.1f}%)")
        results_df.at[idx, 'is_low_quality'] = True
    
    if row['total_cancel_rate'] > LOW_QUALITY_CRITERIA['total_cancel_rate']:
        issues.append(f"High total cancel rate ({row['total_cancel_rate']:.1f}%)")
        results_df.at[idx, 'is_low_quality'] = True
    
    if row['cancel_after_selection_rate'] > LOW_QUALITY_CRITERIA['cancel_after_selection_rate']:
        issues.append(f"High cancel-after-selection rate ({row['cancel_after_selection_rate']:.1f}%)")
        results_df.at[idx, 'is_low_quality'] = True
    
    if row['avg_open_duration_sec'] < LOW_QUALITY_CRITERIA['avg_open_duration_sec']:
        issues.append(f"Short open duration ({row['avg_open_duration_sec']:.1f}s)")
        results_df.at[idx, 'is_low_quality'] = True
    
    if row['final_passage_count'] < LOW_QUALITY_CRITERIA['final_passage_count']:
        issues.append(f"Low passage count ({row['final_passage_count']})")
        results_df.at[idx, 'is_low_quality'] = True
    
    if row['passage_completion_rate'] < LOW_QUALITY_CRITERIA['passage_completion_rate']:
        issues.append(f"Low completion rate ({row['passage_completion_rate']:.1f}%)")
        results_df.at[idx, 'is_low_quality'] = True
    
    results_df.at[idx, 'quality_issues'] = '; '.join(issues)

low_quality_workers = results_df[results_df['is_low_quality'] == True]

print(f"发现 {len(low_quality_workers)} 个低质量工人（占比 {len(low_quality_workers)/len(results_df)*100:.1f}%）")
print()

if len(low_quality_workers) > 0:
    print("低质量工人详情 (Low-Quality Workers Details):")
    print("-" * 80)
    for _, worker in low_quality_workers.iterrows():
        print(f"User ID: {worker['user_id']}")
        print(f"  Issues: {worker['quality_issues']}")
        print(f"  Bot Pass Rate: {worker['bot_pass_rate']:.1f}%")
        print(f"  Easy Negative Click Rate: {worker['easy_neg_rate']:.1f}%")
        print(f"  Total Cancel Rate: {worker['total_cancel_rate']:.1f}%")
        print(f"  Cancel After Selection Rate: {worker['cancel_after_selection_rate']:.1f}%")
        print(f"  Cancel Without Selection Rate: {worker['cancel_without_selection_rate']:.1f}%")
        print(f"  Avg Open Duration: {worker['avg_open_duration_sec']:.2f}s")
        print(f"  Avg Passage Duration: {worker['avg_passage_duration_sec']:.2f}s")
        print(f"  Raw Passage Count: {worker['raw_passage_count']}")
        print(f"  Cancelled Passages: {worker['cancel_after_selection']}")
        print(f"  Final Passage Count: {worker['final_passage_count']}")
        print(f"  Passage Completion Rate: {worker['passage_completion_rate']:.1f}%")
        print()

# 同时输出所有用户的简要统计（英文）
print("=" * 80)
print("All Users Summary (English)")
print("=" * 80)
print()

for _, user in results_df.iterrows():
    quality_status = "⚠️ LOW QUALITY" if user['is_low_quality'] else "✓ Good Quality"
    print(f"User: {user['user_id']} - {quality_status}")
    print(f"  Bot: {user['bot_pass_rate']:.1f}% | Docs: {user['open_doc_count']} | " +
          f"Passages: {user['final_passage_count']}/{user['raw_passage_count']} ({user['passage_completion_rate']:.1f}%)")
    print(f"  Cancel: Total {user['total_cancel_rate']:.1f}% " +
          f"(No-Select: {user['cancel_without_selection_rate']:.1f}%, After-Select: {user['cancel_after_selection_rate']:.1f}%)")
    print(f"  Easy-Neg: {user['easy_neg_rate']:.1f}% | " +
          f"Avg Duration: Open {user['avg_open_duration_sec']:.1f}s, Passage {user['avg_passage_duration_sec']:.1f}s")
    if user['is_low_quality']:
        print(f"  ⚠️  {user['quality_issues']}")
    print()

print()

低质量工人识别

发现 10 个低质量工人（占比 37.0%）

低质量工人详情 (Low-Quality Workers Details):
--------------------------------------------------------------------------------
User ID: 5a9fa90f873cda00010d29d8
  Issues: High easy-neg click rate (2.3%)
  Bot Pass Rate: 100.0%
  Easy Negative Click Rate: 2.3%
  Total Cancel Rate: 23.4%
  Cancel After Selection Rate: 0.0%
  Cancel Without Selection Rate: 23.4%
  Avg Open Duration: 5.06s
  Avg Passage Duration: 6.11s
  Raw Passage Count: 101
  Cancelled Passages: 0
  Final Passage Count: 101
  Passage Completion Rate: 100.0%

User ID: 5589b65afdf99b118adfa595
  Issues: High easy-neg click rate (1.4%)
  Bot Pass Rate: 100.0%
  Easy Negative Click Rate: 1.4%
  Total Cancel Rate: 6.1%
  Cancel After Selection Rate: 0.0%
  Cancel Without Selection Rate: 6.1%
  Avg Open Duration: 7.78s
  Avg Passage Duration: 7.05s
  Raw Passage Count: 142
  Cancelled Passages: 0
  Final Passage Count: 142
  Passage Completion Rate: 100.0%

User ID: 6601a50336e8fbd902820049
  Issues:

In [20]:
# ============================================================================
# 第七步：深入分析特定用户的Easy Negative点击行为
# ============================================================================

def analyze_user_easy_negative_clicks(user_id, filtered_logs, easy_negative_set, 
                                      easy_neg_df):
    """
    深入分析某个用户的Easy Negative点击行为
    
    参数:
    - user_id: 要分析的用户ID
    - filtered_logs: 过滤后的日志数据
    - easy_negative_set: Easy Negative文档集合 (qid, docno)
    - easy_neg_df: Easy Negative文档数据，包含qid, docno, query, text等
    """
    print("=" * 80)
    print(f"用户 {user_id} 的 Easy Negative 点击详细分析")
    print("=" * 80)
    print()
    
    # 获取该用户的所有日志
    user_logs = filtered_logs[filtered_logs['user_id'] == user_id]
    
    # 获取该用户的所有OPEN_DOC事件
    open_doc_logs = user_logs[user_logs['event_type'] == 'OPEN_DOC']
    
    if len(open_doc_logs) == 0:
        print(f"用户 {user_id} 没有打开任何文档")
        return
    
    print(f"用户 {user_id} 总共打开了 {len(open_doc_logs)} 个文档\n")
    
    # 识别Easy Negative点击
    easy_neg_clicks_details = []
    
    for idx, log in open_doc_logs.iterrows():
        qid_str = str(int(log['qid'])) if pd.notna(log['qid']) else '0'
        docno_str = str(log['docno'])
        
        if (qid_str, docno_str) in easy_negative_set:
            # 从easy_neg_df中获取查询和文档信息
            match = easy_neg_df[
                (easy_neg_df['qid'].astype(str) == qid_str) & 
                (easy_neg_df['docno'].astype(str) == docno_str)
            ]
            
            if len(match) > 0:
                query_text = match.iloc[0]['query']
                doc_content = match.iloc[0]['text']
                interleaving_rank = match.iloc[0]['rank'] if 'rank' in match.columns else None
            else:
                query_text = "未找到查询文本"
                doc_content = "未找到文档内容"
                interleaving_rank = None
            
            # 检查是否有passage selection
            passage_logs = user_logs[
                (user_logs['event_type'] == 'PASSAGE_SELECTION') & 
                (user_logs['qid'] == log['qid']) & 
                (user_logs['docno'] == log['docno'])
            ]
            
            # 获取passage selection的详细信息
            passage_details = []
            for _, p_log in passage_logs.iterrows():
                start_idx = p_log['start_idx'] if 'start_idx' in p_log else None
                end_idx = p_log['end_idx'] if 'end_idx' in p_log else None
                if start_idx is not None and end_idx is not None and start_idx >= 0 and end_idx >= 0:
                    passage_details.append({
                        'start': start_idx,
                        'end': end_idx,
                        'duration': p_log['duration'] / 1000 if pd.notna(p_log['duration']) else 0
                    })
            
            # 检查是否有cancel
            cancel_logs = user_logs[
                (user_logs['event_type'] == 'CANCEL_DOC') & 
                (user_logs['qid'] == log['qid']) & 
                (user_logs['docno'] == log['docno'])
            ]
            
            # 检查cancel的类型
            cancel_type = None
            if len(cancel_logs) > 0:
                cancel_log = cancel_logs.iloc[0]
                if cancel_log['start_idx'] == -1 and cancel_log['end_idx'] == -1:
                    cancel_type = "未选择就取消"
                else:
                    cancel_type = "选择后取消"
            
            easy_neg_clicks_details.append({
                'qid': qid_str,
                'docno': docno_str,
                'query': query_text,
                'doc_content': doc_content,
                'duration_sec': log['duration'] / 1000 if pd.notna(log['duration']) else 0,
                'user_rank': log['rank'] if 'rank' in log else None,
                'interleaving_rank': interleaving_rank,
                'has_passage_selection': len(passage_logs) > 0,
                'passage_count': len(passage_logs),
                'passage_details': passage_details,
                'has_cancel': len(cancel_logs) > 0,
                'cancel_type': cancel_type,
                'timestamp': log['timestamp'] if 'timestamp' in log else None
            })
    
    print(f"总共找到 {len(easy_neg_clicks_details)} 个 Easy Negative 点击")
    print(f"Easy Negative点击率: {len(easy_neg_clicks_details)/len(open_doc_logs)*100:.1f}%\n")
    
    # 详细展示每个点击
    if len(easy_neg_clicks_details) > 0:
        for i, click in enumerate(easy_neg_clicks_details, 1):
            print(f"{'='*80}")
            print(f"【点击 {i}/{len(easy_neg_clicks_details)}】")
            print(f"{'='*80}")
            print(f"QID: {click['qid']}")
            print(f"DocNo: {click['docno']}")
            print(f"用户看到的Rank: {click['user_rank']}")
            print(f"Interleaving中的Rank: {click['interleaving_rank']}")
            print(f"查看时长: {click['duration_sec']:.2f} 秒")
            
            # 显示用户行为
            print(f"\n用户行为:")
            print(f"  • 是否选择Passage: {'✓ 是' if click['has_passage_selection'] else '✗ 否'}")
            if click['has_passage_selection']:
                print(f"  • Passage选择次数: {click['passage_count']}")
                for j, p in enumerate(click['passage_details'], 1):
                    print(f"    - 第{j}次: 字符{p['start']}-{p['end']} (选择用时: {p['duration']:.2f}秒)")
            
            print(f"  • 是否取消: {'✓ 是' if click['has_cancel'] else '✗ 否'}")
            if click['has_cancel']:
                print(f"    类型: {click['cancel_type']}")
            
            print(f"\n查询内容:")
            print(f"  「{click['query']}」")
            
            print(f"\n文档内容 (前800字符):")
            print(f"  {'-'*76}")
            doc_preview = click['doc_content'][:800] if isinstance(click['doc_content'], str) else str(click['doc_content'])[:800]
            print(f"  {doc_preview}")
            if len(str(click['doc_content'])) > 800:
                print(f"  ... (还有 {len(str(click['doc_content'])) - 800} 字符)")
            print(f"  {'-'*76}")
            print()
    
    # 汇总统计
    if len(easy_neg_clicks_details) > 0:
        print("=" * 80)
        print("汇总统计")
        print("=" * 80)
        print()
        
        total_duration = sum(c['duration_sec'] for c in easy_neg_clicks_details)
        avg_duration = total_duration / len(easy_neg_clicks_details)
        with_passage = sum(1 for c in easy_neg_clicks_details if c['has_passage_selection'])
        with_cancel = sum(1 for c in easy_neg_clicks_details if c['has_cancel'])
        cancel_without_selection = sum(1 for c in easy_neg_clicks_details 
                                       if c['has_cancel'] and c['cancel_type'] == "未选择就取消")
        cancel_after_selection = sum(1 for c in easy_neg_clicks_details 
                                     if c['has_cancel'] and c['cancel_type'] == "选择后取消")
        
        print(f"Easy Negative点击总数: {len(easy_neg_clicks_details)}")
        print(f"占所有文档点击的比例: {len(easy_neg_clicks_details)/len(open_doc_logs)*100:.1f}%")
        print()
        print(f"平均查看时长: {avg_duration:.2f} 秒")
        print(f"总查看时长: {total_duration:.2f} 秒")
        print()
        print(f"有Passage选择的点击: {with_passage} ({with_passage/len(easy_neg_clicks_details)*100:.1f}%)")
        print(f"有取消行为的点击: {with_cancel} ({with_cancel/len(easy_neg_clicks_details)*100:.1f}%)")
        if with_cancel > 0:
            print(f"  - 未选择就取消: {cancel_without_selection} ({cancel_without_selection/with_cancel*100:.1f}%)")
            print(f"  - 选择后取消: {cancel_after_selection} ({cancel_after_selection/with_cancel*100:.1f}%)")
        print()
        
        # 按查询分组统计
        from collections import defaultdict
        query_stats = defaultdict(lambda: {
            'count': 0, 
            'query': None,
            'with_passage': 0,
            'with_cancel': 0,
            'total_duration': 0
        })
        
        for click in easy_neg_clicks_details:
            qid = click['qid']
            query_stats[qid]['count'] += 1
            query_stats[qid]['query'] = click['query']
            query_stats[qid]['total_duration'] += click['duration_sec']
            if click['has_passage_selection']:
                query_stats[qid]['with_passage'] += 1
            if click['has_cancel']:
                query_stats[qid]['with_cancel'] += 1
        
        print("按查询分组:")
        print("-" * 80)
        for qid, stats in sorted(query_stats.items(), key=lambda x: x[1]['count'], reverse=True):
            print(f"QID {qid}: {stats['count']} 次点击")
            print(f"  查询: {stats['query']}")
            print(f"  平均时长: {stats['total_duration']/stats['count']:.2f}秒")
            print(f"  有Passage: {stats['with_passage']}/{stats['count']}")
            print(f"  有取消: {stats['with_cancel']}/{stats['count']}")
            print()
        
        # 质量评估
        print("=" * 80)
        print("质量评估")
        print("=" * 80)
        print()
        
        # 基于多个指标评估质量
        quality_score = 0
        quality_factors = []
        
        # 1. Passage选择率 (高=好)
        passage_rate = with_passage / len(easy_neg_clicks_details)
        if passage_rate > 0.5:
            quality_factors.append("✓ 高Passage选择率 (>50%)")
            quality_score += 2
        elif passage_rate > 0.2:
            quality_factors.append("• 中等Passage选择率 (20-50%)")
            quality_score += 1
        else:
            quality_factors.append("✗ 低Passage选择率 (<20%)")
        
        # 2. 平均查看时长 (适中=好，太短=差，太长也可能有问题)
        if 5 <= avg_duration <= 60:
            quality_factors.append("✓ 合理的查看时长 (5-60秒)")
            quality_score += 2
        elif 2 <= avg_duration < 5:
            quality_factors.append("• 较短的查看时长 (2-5秒)")
            quality_score += 1
        else:
            quality_factors.append("✗ 异常的查看时长 (<2秒或>60秒)")
        
        # 3. 取消率 (低=好)
        cancel_rate = with_cancel / len(easy_neg_clicks_details)
        if cancel_rate < 0.3:
            quality_factors.append("✓ 低取消率 (<30%)")
            quality_score += 2
        elif cancel_rate < 0.5:
            quality_factors.append("• 中等取消率 (30-50%)")
            quality_score += 1
        else:
            quality_factors.append("✗ 高取消率 (>50%)")
        
        print("评估因素:")
        for factor in quality_factors:
            print(f"  {factor}")
        print()
        
        print(f"质量分数: {quality_score}/6")
        if quality_score >= 5:
            quality_level = "高质量 ✓"
            print(f"评估结果: {quality_level}")
            print("用户的Easy Negative点击行为显示出较高的参与度和判断质量")
        elif quality_score >= 3:
            quality_level = "中等质量 •"
            print(f"评估结果: {quality_level}")
            print("用户的Easy Negative点击行为质量一般，需进一步检查")
        else:
            quality_level = "低质量 ✗"
            print(f"评估结果: {quality_level}")
            print("用户的Easy Negative点击行为可能存在质量问题")
        
        return {
            'user_id': user_id,
            'total_clicks': len(easy_neg_clicks_details),
            'avg_duration': avg_duration,
            'passage_rate': passage_rate,
            'cancel_rate': cancel_rate,
            'quality_score': quality_score,
            'quality_level': quality_level
        }
    else:
        print("该用户没有点击Easy Negative文档")
        return None


# ============================================================================
# 使用示例
# ============================================================================
print("\n\n")
print("=" * 80)
print("开始分析目标用户的Easy Negative点击行为")
print("=" * 80)
print("\n")

# 分析单个用户
target_user_id = '66ac261efa487682e38c3024'

result = analyze_user_easy_negative_clicks(
    user_id=target_user_id,
    filtered_logs=filtered_logs,
    easy_negative_set=easy_negative_set,
    easy_neg_df=easy_neg_df
)





开始分析目标用户的Easy Negative点击行为


用户 66ac261efa487682e38c3024 的 Easy Negative 点击详细分析

用户 66ac261efa487682e38c3024 总共打开了 189 个文档

总共找到 7 个 Easy Negative 点击
Easy Negative点击率: 3.7%

【点击 1/7】
QID: 833860
DocNo: 4620277
用户看到的Rank: None
Interleaving中的Rank: 8
查看时长: 0.87 秒

用户行为:
  • 是否选择Passage: ✓ 是
  • Passage选择次数: 1
    - 第1次: 字符260-367 (选择用时: 4.54秒)
  • 是否取消: ✗ 否

查询内容:
  「what is the most popular food in switzerland」

文档内容 (前800字符):
  ----------------------------------------------------------------------------
  Queen of the mountains. The Rigi mountain is probably one of the most popular day trips in Switzerland and can be explored at any time of year. In summer, the Rigi offers 120 kilometres of hiking paths – ranging from an easy stroll to an Alpine mountain trek. Those who like to take things a little easier can go up the mountain by train and enjoy the fabulous views.
  ----------------------------------------------------------------------------

【点击 2/7】
QID: 915593
DocNo: 98515
用户看到的